# Preprocessing — NYC Rideshare Earnings

Cleans the raw NYC TLC FHVHV trip data, engineers temporal and
location features, joins external weather + holiday data, and writes
month-partitioned curated Parquet files to `data/curated/`.

All logic lives in `scripts/preprocessing.py` and `scripts/io_utils.py`
— this notebook is the narrative layer that orchestrates the pipeline
and surfaces intermediate sanity checks.

In [ ]:
import sys
sys.path.append('../scripts')

from spark import get_spark
from io_utils import (
    load_raw_fhvhv, load_weather, load_holidays, write_curated,
)
from preprocessing import (
    drop_unused_columns, filter_to_uber_lyft, filter_date_range,
    filter_valid_pulocation, apply_baseline_filters,
    encode_categorical, add_time_features, remove_outliers,
    standardise, add_earnings_columns,
    clean_weather, join_weather, add_holiday_flag,
)
from functions import get_dataset_shape, get_missing_value_counts

spark = get_spark()

## 1. Load & Inspect Raw FHVHV

In [ ]:
fhvhv_df = load_raw_fhvhv(spark, '../data/raw')
original_count = fhvhv_df.count()
get_dataset_shape(fhvhv_df)
fhvhv_df.printSchema()

## 2. Filtering & Cleaning

Drop irrelevant columns, restrict to Uber (HV0003) and Lyft (HV0005),
clip to May–Nov 2023, and constrain pickup-location IDs to the valid
TLC range (1–263).

In [ ]:
fhvhv_df = (
    fhvhv_df
    .transform(drop_unused_columns)
    .transform(filter_to_uber_lyft)
)
fhvhv_df = filter_date_range(fhvhv_df, 'pickup_datetime', '2023-05-01', '2023-12-01')
fhvhv_df = filter_valid_pulocation(fhvhv_df)
get_dataset_shape(fhvhv_df)

## 3. Feature Engineering

One-hot encode license, derive day/hour/month from the pickup timestamp,
then one-hot encode day-of-week, hour-of-day, and pickup zone.

Sanity check: May 1st 2023 was a Monday → `day_of_week = 2` in PySpark's
encoding (Sunday = 1).

In [ ]:
fhvhv_df = encode_categorical(fhvhv_df, 'hvfhs_license_num', 'license_vec', 'license_index')
fhvhv_df = add_time_features(fhvhv_df)
fhvhv_df = encode_categorical(fhvhv_df, 'day_of_week', 'day_vec', 'day_index')
fhvhv_df = encode_categorical(fhvhv_df, 'hour_of_day', 'hour_vec', 'hour_index')
fhvhv_df = encode_categorical(fhvhv_df, 'PULocationID', 'PULocation_vec', 'PULocation_index')
get_dataset_shape(fhvhv_df)

## 4. Outlier Removal & Standardisation

Apply baseline filters (positive trip distance, ≥1min duration,
non-negative tips, ≥$3 driver pay), then the project's domain-aware
IQR rule (`(√log(N) − 0.5) × IQR`) — more permissive than 1.5×IQR so
high-value airport/long-haul trips aren't over-filtered.

Earnings and earnings-per-hour are derived after pay/tip filtering,
then put through the same outlier rule as a second pass.

In [ ]:
fhvhv_df = apply_baseline_filters(fhvhv_df)
fhvhv_df = remove_outliers(fhvhv_df, ['trip_miles', 'trip_time', 'tips', 'driver_pay'])
fhvhv_df = standardise(fhvhv_df, ['trip_miles', 'trip_time'])
fhvhv_df = add_earnings_columns(fhvhv_df)
fhvhv_df = remove_outliers(fhvhv_df, ['earnings', 'earnings_per_hour'])

fhvhv_df = fhvhv_df.drop(
    'tips', 'driver_pay', 'license_index', 'day_index', 'hour_index', 'PULocation_index',
)

remaining = fhvhv_df.count() / original_count
print(f'Rows remaining: {remaining:.1%} of original')

## 5. External Data: Weather

NULL `preciptype` is treated as no precipitation (a logical assumption —
it isn't always raining or snowing). IQR outlier removal is intentionally
not applied to weather variables: observed ranges (e.g. coldest 2023 NYC
feels-like ≈ 6.8°F) are plausible, and dropping rows would create gaps
in the join downstream.

In [ ]:
weather_df = load_weather(
    spark, '../data/raw_csv/NYC_hourly_weather_2023-05-01_to_2024-05-31.csv',
)
get_missing_value_counts(weather_df)

weather_df = clean_weather(weather_df)
weather_df = filter_date_range(weather_df, 'datetime', '2023-05-01', '2023-12-01')
weather_df = standardise(weather_df, ['feelslike', 'precip'])
weather_df = encode_categorical(weather_df, 'preciptype', 'preciptype_vec', 'preciptype_index')
weather_df = weather_df.drop('preciptype_index')
get_dataset_shape(weather_df)

## 6. External Data: Public Holidays

In [ ]:
holidays_df = load_holidays(
    spark,
    '../data/raw_csv/2023_holidays.csv',
    '../data/raw_csv/2024_holidays.csv',
)
holiday_dates = [row['Date'] for row in holidays_df.select('Date').collect()]
print(f'{len(holiday_dates)} holiday dates loaded')

## 7. Join & Write Curated Chunks

Left-join weather onto trips on `pickup_hour == datetime`, add the
holiday flag, drop linking columns, and write one Parquet file per
month to `data/curated/`. The modelling and analysis notebooks read
these back with a glob pattern.

In [ ]:
final_df = join_weather(fhvhv_df, weather_df)
final_df = add_holiday_flag(final_df, holiday_dates)

final_df = final_df.drop(
    'pickup_hour', 'pickup_date', 'datetime', 'pickup_datetime',
    'precip', 'preciptype', 'trip_time',
)
get_dataset_shape(final_df)
final_df.printSchema()

In [ ]:
write_curated(final_df, '../data/curated', months=[5, 6, 7, 8, 9, 10, 11])